# Baby Step 7 — Counterparty Selection and Outreach Readiness

This transparent notebook ranks a **synthetic** universe of 12 financial investors and 7 strategic counterparties against the approved VoltEdge preferred-equity structure. It applies fit scores and hard gates separately, tests ranking sensitivity, and returns a bounded committee decision.

> **Control boundary:** this notebook does not contact anyone and does not authorize outreach.


## The control question

Which synthetic counterparties best fit the approved $32m staged nonparticipating preferred-equity structure, and which candidates must be held or excluded before any future outreach?


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

VAULT_NAME = "Alejandro-Reynoso-Investment-Banking-Vault"
env_path = os.environ.get("VAULT_PATH")
possible = ([Path(env_path)] if env_path else []) + [
    Path.cwd() / VAULT_NAME,
    Path("/content/drive/MyDrive") / VAULT_NAME,
    Path("/workspace/scratch/9ba1ff46ede5") / VAULT_NAME,
]
VAULT = next((p for p in possible if str(p) and p.exists()), None)
if VAULT is None:
    raise FileNotFoundError("Set VAULT_PATH or place the vault in the current directory / Google Drive.")
DRY_RUN = True
print("Vault:", VAULT)
print("DRY_RUN:", DRY_RUN)


## 1. Resolve and validate the input contract

The scoring notebook depends on the approved structure, the 19-party universe, the conflict screen, and the governed evidence registers.


In [ ]:
required = [
    VAULT / "Data" / "loop_007_counterparty_universe.csv",
    VAULT / "Data" / "loop_007_counterparty_scores.csv",
    VAULT / "Data" / "loop_007_conflict_screen.csv",
    VAULT / "Data" / "claim_register.csv",
    VAULT / "Data" / "source_registry.csv",
    VAULT / "Data" / "loop_006_transaction_structures.csv",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing inputs: {missing}"
print(f"Validated {len(required)} governed inputs.")


## 2. Load the counterparty and governance layers


In [ ]:
universe = pd.read_csv(required[0])
stored_scores = pd.read_csv(required[1])
conflicts = pd.read_csv(required[2])
claims = pd.read_csv(required[3])
sources = pd.read_csv(required[4])
structures = pd.read_csv(required[5])

assert len(universe) == 19
assert universe["counterparty_type"].value_counts().to_dict() == {"Financial investor": 12, "Strategic counterparty": 7}
print("Universe:", universe["counterparty_type"].value_counts().to_dict())
display(universe[["counterparty_id","name","counterparty_type","minimum_check_usd_m","maximum_check_usd_m","conflict_status"]])


## 3. Reconstruct the approved transaction baseline

Counterparties are selected for the structure already approved in DEC-006. Step 7 does not reopen the capital-structure decision.


In [ ]:
preferred = structures.loc[structures["option_id"].eq("STR-002")].iloc[0]
baseline = {
    "structure": preferred["structure"],
    "total_funding_usd_m": float(preferred["total_funding_usd_m"]),
    "upfront_funding_usd_m": float(preferred["upfront_funding_usd_m"]),
    "milestone_tranche_usd_m": float(preferred["milestone_tranche_usd_m"]),
    "pik_pct": float(preferred["pik_pct"]),
    "liquidation_preference_x": float(preferred["liquidation_preference_x"]),
}
assert baseline["total_funding_usd_m"] == 32
assert baseline["upfront_funding_usd_m"] == 24
print(json.dumps(baseline, indent=2))


## 4. Recalculate the weighted fit score

Each rating is 1–5. Weights total 100. The score supports comparison; it does not decide eligibility.


In [ ]:
weights = {
    "structure_fit": 18, "check_size_fit": 14, "sector_expertise": 14,
    "return_fit": 10, "risk_appetite": 10, "governance_fit": 8,
    "strategic_value": 8, "relationship_strength": 10, "timing_readiness": 8,
}
assert sum(weights.values()) == 100
recalc = universe.copy()
for field, weight in weights.items():
    recalc[field + "_points"] = recalc[field] * weight / 5
recalc["recalculated_score"] = recalc[[f + "_points" for f in weights]].sum(axis=1).round(1)
merged = recalc.merge(stored_scores[["counterparty_id","total_fit_score"]], on="counterparty_id")
assert np.allclose(merged["recalculated_score"], merged["total_fit_score"])
print("All stored scores reproduced exactly.")
display(merged[["counterparty_id","name","recalculated_score"]].sort_values("recalculated_score", ascending=False))


## 5. Apply hard gates outside the score

A high score cannot override a hard conflict, insufficient check size, structure mismatch, unusable timing, or a control requirement.


In [ ]:
gate_view = universe[["counterparty_id","name","hard_gate_failed","conflict_status","outreach_wave","selection_rationale"]].copy()
assert gate_view["hard_gate_failed"].sum() == 6
assert (gate_view.loc[gate_view["hard_gate_failed"], "outreach_wave"] == "Excluded").all()
assert not ((gate_view["hard_gate_failed"]) & (gate_view["outreach_wave"] == "Wave 1")).any()
print("Hard-gate exclusions:", int(gate_view["hard_gate_failed"].sum()))
display(gate_view.loc[gate_view["hard_gate_failed"]])


## 6. Produce the ranked funnel and outreach waves


In [ ]:
ranking = stored_scores.sort_values(["outreach_wave","total_fit_score"], ascending=[True,False]).copy()
wave_order = ["Wave 1","Wave 2","Reserve","Excluded"]
counts = ranking["outreach_wave"].value_counts().reindex(wave_order).fillna(0).astype(int)
assert counts.to_dict() == {"Wave 1": 6, "Wave 2": 3, "Reserve": 4, "Excluded": 6}
print(counts.to_dict())

fig, ax = plt.subplots(figsize=(9,4.5))
colors = ["#17324D","#2E74B5","#7A8A99","#B34A4A"]
ax.bar(counts.index, counts.values, color=colors)
ax.set_title("Counterparty selection funnel")
ax.set_ylabel("Synthetic counterparties")
ax.spines[["top","right"]].set_visible(False)
for i,v in enumerate(counts.values): ax.text(i,v+.15,str(v),ha="center",weight="bold")
plt.show()


## 7. Inspect Wave 1 and conditional Wave 2


In [ ]:
shortlist = ranking.loc[ranking["outreach_wave"].isin(["Wave 1","Wave 2"]), [
    "counterparty_id","name","counterparty_type","total_fit_score","conflict_status","outreach_wave","selection_rationale"
]].copy()
shortlist["wave_order"] = shortlist["outreach_wave"].map({"Wave 1":1,"Wave 2":2})
shortlist = shortlist.sort_values(["wave_order","total_fit_score"], ascending=[True,False]).drop(columns="wave_order")
display(shortlist)
assert shortlist.query('outreach_wave == "Wave 1"').shape[0] == 6
assert shortlist.query('outreach_wave == "Wave 2"').shape[0] == 3


## 8. Review conflict and information-control conditions


In [ ]:
attention = conflicts.loc[(conflicts["conflict_status"] != "Clear") | (conflicts["hard_gate_failed"] == True)]
display(attention[["counterparty_id","name","conflict_status","issue","required_resolution"]])
wave1_ids = set(shortlist.query('outreach_wave == "Wave 1"')["counterparty_id"])
assert not set(conflicts.query('conflict_status == "Hard conflict"')["counterparty_id"]) & wave1_ids
print("No hard-conflict party appears in Wave 1.")


## 9. Test sensitivity to relationship and strategic value

This alternative increases relationship and strategic-value weights while reducing structure and check-size weights. The purpose is to see whether the short list is fragile.


In [ ]:
alt_weights = {
    "structure_fit": 14, "check_size_fit": 10, "sector_expertise": 14,
    "return_fit": 8, "risk_appetite": 8, "governance_fit": 8,
    "strategic_value": 14, "relationship_strength": 16, "timing_readiness": 8,
}
assert sum(alt_weights.values()) == 100
alt = universe.copy()
alt["alt_score"] = sum(alt[k] * w / 5 for k,w in alt_weights.items())
eligible = alt.loc[~alt["hard_gate_failed"]].copy()
base_order = stored_scores.loc[stored_scores["eligible"]].set_index("counterparty_id")["total_fit_score"]
comparison = eligible.set_index("counterparty_id")[["alt_score"]].join(base_order).dropna()
rho = comparison.corr(method="spearman").iloc[0,1]
base_top6 = set(base_order.nlargest(6).index)
alt_top6 = set(comparison["alt_score"].nlargest(6).index)
print("Spearman rank correlation:", round(rho,3))
print("Top-six retention:", len(base_top6 & alt_top6), "of 6")
assert rho > 0.80
assert len(base_top6 & alt_top6) >= 5


## 10. Run the Step 7 decision engine


In [ ]:
checks = {
    "approved_structure_preserved": baseline["structure"] == "Staged nonparticipating preferred equity",
    "wave_1_has_six": counts["Wave 1"] == 6,
    "wave_2_has_three": counts["Wave 2"] == 3,
    "six_hard_gate_exclusions": counts["Excluded"] == 6,
    "no_hard_conflict_in_wave_1": not bool(set(conflicts.query('conflict_status == "Hard conflict"')["counterparty_id"]) & wave1_ids),
    "ranking_is_robust": rho > 0.80 and len(base_top6 & alt_top6) >= 5,
    "evidence_register_advanced": len(sources) == 12 and len(claims) == 24,
}
assert all(checks.values()), checks
recommendation = "APPROVE INTERNAL SHORT LIST — NO OUTREACH"
print({key: bool(value) for key, value in checks.items()})
print("Recommendation:", recommendation)


## 11. Validate the authorization boundary


In [ ]:
permitted = [
    "Finalize internal ranking", "Design recipient verification", "Draft synthetic disclosure tiers",
    "Define conflict clearance", "Prepare a new committee gate",
]
prohibited = [
    "Contact a counterparty", "Send information", "Communicate valuation or terms",
    "Solicit an indication", "Negotiate", "Execute",
]
boundary = pd.DataFrame({"Permitted now": pd.Series(permitted), "Still prohibited": pd.Series(prohibited)})
display(boundary)


## 12. Optional governed write

The default remains dry run. Set `DRY_RUN = False` only after reviewing the output and only to write a reproduction manifest inside the vault.


In [ ]:
manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "loop": "007",
    "recommendation": recommendation,
    "wave_counts": counts.to_dict(),
    "sensitivity_spearman": round(float(rho),3),
    "top_six_retention": len(base_top6 & alt_top6),
    "external_action_authorized": False,
}
if DRY_RUN:
    print("Dry run: no files written.")
    print(json.dumps(manifest, indent=2))
else:
    out = VAULT / "Reports" / "Counterparty Ranking Reproductions"
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"loop_007_manifest_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("Wrote", path)


## 13. Define the Baby Step 8 input contract

Baby Step 8 should consume the approved short list and create recipient verification, conflict clearance, disclosure tiers, approved messaging, contact logs, response capture, and a final human outreach gate.


In [ ]:
next_contract = pd.DataFrame([
    ["Approved internal short list", "6 Wave 1 + 3 conditional Wave 2", "DEC-007"],
    ["Recipient verification", "Identity and authority confirmation", "Required before contact"],
    ["Conflict clearance", "Refreshed matter-level screen", "Required before contact"],
    ["Disclosure control", "Teaser / NDA / CIM tiers", "Human approval"],
    ["Outreach decision", "Go / hold / reject", "New committee gate"],
], columns=["Input", "Definition", "Control"] )
display(next_contract)


## What Baby Step 7 demonstrates

The exo-brain can now transform an approved structure into a governed counterparty plan:

**Structure → candidate universe → weighted fit → hard gates → shortlist → conflict conditions → human permission.**

The result is decision-ready, reproducible and still unable to outrun its authority.
